# LAB 3 — Optimize Tokens: prefix cache (เร็วขึ้นเมื่อถามซ้ำ)
ส่ง context เดิม 2 รอบ → รอบ 2 เร็วขึ้นเพราะถูกแคช · กด `Shift+Enter`

In [ ]:
import os, time
from openai import OpenAI
client = OpenAI(base_url=os.environ['OPENAI_BASE_URL'], api_key='EMPTY'); MODEL=os.environ['MODEL']
LONG = 'You are a DGX Spark assistant. Context: ' + ('DGX Spark = GB10, 128GB unified memory. ' * 30)
def ttft(q):
    t=time.time(); first=None
    s=client.chat.completions.create(model=MODEL,max_tokens=24,stream=True,messages=[{'role':'system','content':LONG},{'role':'user','content':q}])
    for c in s:
        if c.choices and c.choices[0].delta.content and first is None: first=time.time()-t
    return first
print('prefix ~', len(LONG)//4, 'tokens (เหมือนกันทั้ง 2 รอบ)')

In [ ]:
a = ttft('สรุป DGX Spark 1 ประโยค'); print(f'รอบ 1 (cold): TTFT = {a*1000:.0f} ms')
b = ttft('สรุป DGX Spark 1 ประโยค'); print(f'รอบ 2 (warm): TTFT = {b*1000:.0f} ms')
print(f'\n✅ เร็วขึ้น {(1-b/a)*100:.0f}% — prefix เดิมถูกแคช ไม่ต้องคิดใหม่ = ถูกลง/เร็วขึ้น' if b<a else 'ลองรันรอบ 3 อีกที')

**บทเรียน:** ของถูกที่สุดคือ token ที่ไม่ต้องสร้างใหม่ → **cost per token = KPI** (quantize เล็ก, แคชเยอะ, อย่ายัด context เกิน)